In [ ]:
# Path config (auto-injected). Loads WORK_DIR from .env so notebooks run anywhere.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd().resolve()
for _ in range(8):
    if (_p / '.env').is_file():
        load_dotenv(_p / '.env')
        break
    if _p == _p.parent:
        break
    _p = _p.parent

WORK_DIR = os.environ['WORK_DIR']
RETRIEVAL_DIR = f"{WORK_DIR}/retrieval_assets"

In [ ]:
!pip install lettucedetect

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:



import json
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import pickle
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STEP 1: Load Dataset
# ============================================================

DATASET_PATH = f'{WORK_DIR}/gemma/hotpot/hotpot_qwen25_rag_length_controlled_1000_binmatched.json'
YOUR_RESULTS_PATH = f'{WORK_DIR}/gemma/hotpot/tests/10/super/hotpot_sds_perplexity_results.pkl'  # UPDATE THIS PATH

with open(DATASET_PATH) as f:
    dataset = json.load(f)

labels = np.array([s['hallucinated'] for s in dataset])
print(f"Loaded {len(dataset)} samples")
print(f"  Hallucinated: {labels.sum()}")
print(f"  Non-hallucinated: {(1-labels).sum()}")

# Load YOUR results for fair comparison
print(f"\nLoading your results from {YOUR_RESULTS_PATH}...")
with open(YOUR_RESULTS_PATH, 'rb') as f:
    your_results = pickle.load(f)

# Extract your combined scores (SDS + Uncertainty)
if 'results' in your_results and 'combined' in your_results['results']:
    your_scores = your_results['results']['combined']['scores']
elif 'combined_scores' in your_results:
    your_scores = your_results['combined_scores']
else:
    # Fallback: compute from components
    from sklearn.linear_model import LogisticRegression
    X = np.column_stack([
        your_results['SDS'],
        your_results['perplexity'],
        your_results['mean_prob']
    ])
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X, your_results['labels'])
    your_scores = clf.predict_proba(X)[:, 1]

your_auroc = roc_auc_score(your_results['labels'], your_scores)
print(f"Your method AUROC: {your_auroc:.4f}")
print(f"Dataset size match: {len(your_results['labels'])} vs {len(labels)}")

# ============================================================
# STEP 2: Load LettuceDetect
# ============================================================

print("\nLoading LettuceDetect model...")

from lettucedetect.models.inference import HallucinationDetector

# Load the model (uses ModernBERT under the hood)
# Options: "lettucedect-base-modernbert-en-v1" (smaller) or "lettucedect-large-modernbert-en-v1" (better)
detector = HallucinationDetector(
    method="transformer",
    model_path="KRLabsOrg/lettucedect-large-modernbert-en-v1"
)

print("LettuceDetect loaded!")

# ============================================================
# STEP 3: Run Detection
# ============================================================

print("\nRunning LettuceDetect on dataset...")

hallucination_scores = []
predictions = []
processed_indices = []  # Track which samples were successfully processed

for idx, sample in enumerate(tqdm(dataset)):
    question = sample['question']
    context = sample['context']
    answer = sample['answer']

    # Format context as LIST (LettuceDetect expects list of strings)
    if isinstance(context, list):
        context_list = context
    else:
        context_list = [context]

    # FIX #3: Use full context to match your method's input
    # ModernBERT supports up to 8192 tokens (~32k chars)
    # Keep all passages, truncate each to 8000 chars to stay within limits
    context_list = [c[:8000] for c in context_list]  # Much more generous than before

    try:
        # LettuceDetect expects: context (list), question, answer, output_format
        result = detector.predict(
            context=context_list,
            question=question,
            answer=answer,
            output_format="spans"
        )

        # DEBUG: Print result format on first few samples
        if len(hallucination_scores) < 3:
            print(f"Result type: {type(result)}")
            print(f"Result: {result}")

        # LettuceDetect returns token-level predictions as a list of dicts
        # Empty list = no hallucination detected
        # Non-empty list = hallucination spans detected

        if isinstance(result, list):
            if len(result) == 0:
                # No hallucination detected
                score = 0.0
            else:
                # FIX #1: Use MEAN confidence instead of MAX
                # This prevents single high-confidence tokens from dominating
                confidences = []
                for span in result:
                    if isinstance(span, dict):
                        conf = span.get('confidence', span.get('score', span.get('probability', 1.0)))
                        if isinstance(conf, (int, float)):
                            confidences.append(conf)
                score = np.mean(confidences) if confidences else 0.0  # Changed from max() to mean()
        elif isinstance(result, dict):
            score = result.get('hallucination_score',
                              result.get('score',
                              result.get('confidence', 0.5)))
        elif isinstance(result, (int, float)):
            score = float(result)
        else:
            score = 0.5

        hallucination_scores.append(score)

        # FIX #2: Align binary prediction with score threshold
        predictions.append(1 if score > 0.5 else 0)  # Use score, not span existence

        # Track this sample as successfully processed
        processed_indices.append(idx)

    except Exception as e:
        # FIX #5: Skip error samples instead of injecting 0.5
        # This prevents biasing the ROC curve with neutral scores
        print(f"\nError on sample {idx}: {e}")
        print(f"Skipping this sample...")
        continue  # Skip this sample entirely

hallucination_scores = np.array(hallucination_scores)
predictions = np.array(predictions)
processed_indices = np.array(processed_indices)

print(f"\nProcessed {len(hallucination_scores)} samples successfully")
print(f"Skipped {len(dataset) - len(hallucination_scores)} samples due to errors")

# ============================================================
# STEP 4: Evaluate
# ============================================================

# Match labels to successfully processed samples using tracked indices
labels_processed = labels[processed_indices]

# Also subset your results to match the same samples
your_scores_matched = your_scores[processed_indices]
your_labels_matched = your_results['labels'][processed_indices]

# Verify alignment
assert len(labels_processed) == len(hallucination_scores), "Label mismatch!"
assert np.array_equal(labels_processed, your_labels_matched), "Label alignment error!"

if len(hallucination_scores) < len(labels):
    print("\nWARNING: Some samples were skipped. Results are on subset only.")
    print(f"Comparing on {len(hallucination_scores)} shared samples.")

# Recompute your AUROC on the matched subset
your_auroc_matched = roc_auc_score(your_labels_matched, your_scores_matched)

print("\n" + "="*60)
print("LETTUCEDETECT RESULTS ON HOTPOTQA (CORRECTED)")
print("="*60)

auroc = roc_auc_score(labels_processed, hallucination_scores)
f1 = f1_score(labels_processed, predictions)
precision = precision_score(labels_processed, predictions)
recall = recall_score(labels_processed, predictions)

print(f"\nAUROC:     {auroc:.4f}")
print(f"F1:        {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

# FIX #4: Compare against YOUR results computed on SAME data
print("\n" + "-"*60)
print("COMPARISON ON HOTPOTQA (SAME SAMPLES)")
print("-"*60)
print(f"LettuceDetect (corrected):  {auroc:.4f}")
print(f"Your method (SDS+Unc):      {your_auroc_matched:.4f}")
if auroc < your_auroc_matched:
    improvement = your_auroc_matched - auroc
    pct_improvement = (improvement / auroc) * 100
    print(f"Your improvement:           +{improvement:.4f} ({pct_improvement:.1f}%)")
else:
    print(f"LettuceDetect better:       +{auroc - your_auroc_matched:.4f}")

print("\n" + "-"*60)
print("ORIGINAL YOUR METHOD (FULL DATASET)")
print("-"*60)
print(f"Your AUROC (all samples):   {your_auroc:.4f}")
print(f"Dataset size:               {len(your_results['labels'])} samples")

# ============================================================
# STEP 5: Save Results
# ============================================================

results = {
    'hallucination_scores': hallucination_scores,
    'predictions': predictions,
    'labels': labels_processed,
    'processed_indices': processed_indices,
    'auroc': auroc,
    'f1': f1,
    'precision': precision,
    'recall': recall,
    'your_auroc_full': your_auroc,
    'your_auroc_matched': your_auroc_matched,
    'your_scores_matched': your_scores_matched,
    'num_processed': len(hallucination_scores),
    'num_total': len(dataset),
    'num_skipped': len(dataset) - len(hallucination_scores),
}

with open('lettucedetect_hotpotqa_results_corrected.pkl', 'wb') as f:
    pickle.dump(results, f)

print("\n" + "="*60)
print("Results saved to lettucedetect_hotpotqa_results_corrected.pkl")
print("="*60)

print("\nSUMMARY OF CORRECTIONS APPLIED:")
print("1. ✅ Score extraction: mean(confidences) instead of max()")
print("2. ✅ Binary predictions: aligned with score threshold (0.5)")
print("3. ✅ Context truncation: 8000 chars per passage (vs 3000 before)")
print("4. ✅ Comparison: loaded your results from same dataset")
print("5. ✅ Error handling: skipped samples instead of injecting 0.5")
print("\nExpected impact: AUROC will be LOWER but MORE ACCURATE")
